# Pipeline 1: YOLOv10-Nano + Depth Anything V2 (Metric Outdoor)
**Owner:** Member 1 (Nehir) | **Branch:** `pipeline-1`

This notebook runs **only Pipeline 1** steps:
1. Clones your branch from GitHub and mounts your Google Drive.
2. Downloads the official KITTI dataset and runs `01_prepare_data.py` (preserving locked `data/splits/*.txt` if present).
3. Fine-tunes `YOLOv10-Nano` across 3 random seeds (`42, 123, 999`).
4. Backs up trained weights (`checkpoints/yolov10n_kitti_seed*.pt`) to Google Drive.
5. Evaluates Pipeline 1 with Depth Anything V2 Metric Outdoor and Bounding-Box filtering ablation.

In [ ]:
!nvidia-smi
from google.colab import drive
import os

drive.mount('/content/drive')
SHARED_DRIVE = '/content/drive/MyDrive/CS423_Shared_Project'
os.makedirs(f'{SHARED_DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{SHARED_DRIVE}/results', exist_ok=True)
print('Google Drive backup folder ready:', SHARED_DRIVE)

In [ ]:
REPO_URL = "https://github.com/nehirvardar/cs423-assistive-navigation.git"
BRANCH = "pipeline-1"  # Your branch name

!rm -rf /content/project
!git clone -b {BRANCH} {REPO_URL} /content/project
%cd /content/project
!pip install -q -r requirements.txt
!apt-get update -qq && apt-get install -qq -y aria2

In [ ]:
# Download and prepare KITTI dataset
# (If you only want to train YOLO first, you can pass --skip-velodyne;
# full download including Velodyne LiDAR is required for depth evaluation)
!python scripts/download_kitti.py
!python scripts/01_prepare_data.py

In [ ]:
# Fine-tune YOLOv10-Nano across 3 random seeds
!python scripts/03_train_yolo.py --model yolov10n --seeds 42 123 999 --epochs 50 --batch 16 --device 0

# Back up trained weights to Google Drive
!cp -v checkpoints/yolov10n_kitti_seed*.pt {SHARED_DRIVE}/checkpoints/

In [ ]:
# Quick verification for Pipeline 1 (first 50 test frames with Depth Anything V2 + BBox Ablation)
!python scripts/04_evaluate_all.py --split test --max-frames 50 --device cuda
!cp -r results {SHARED_DRIVE}/